# joins and data sources, nba_demo_test.ipynb

Where every base table came from (endpoint + params), then every merge in the main notebook. Code cells are copy pasted from source, they won't run standalone here since nothing's loaded in this notebook.

## how the data was pulled

All of this actually runs in `nba_graphing_data_more_players.ipynb`, cached to parquet, then just loaded with `pd.read_parquet(...)` at the top of `nba_demo_test.ipynb`. Listing it here since "how was this table made" is a different question than "how do tables join."

`player_stats_advanced_df`, `player_stats_basic_df`: `leaguedashplayerstats.LeagueDashPlayerStats`, one call per season, looped across 22 seasons (2004 05 through 2025 26). `measure_type_detailed_defense` is `'Advanced'` for one table and `'Base'` for the other, that's the only difference in the call. `season_type_all_star='Regular Season'`.

`clutch_adv_df`: `leaguedashplayerclutch.LeagueDashPlayerClutch`, same season loop, `measure_type_detailed_defense='Advanced'`, `clutch_time='Last 5 Minutes'`, `ahead_behind='Ahead or Behind'`, `point_diff=5`.

`home_clutch_df` / `road_clutch_df`: identical clutch call to the one above, just with `location_nullable='Home'` or `'Road'` added.

`all_awards_df`: `playerawards.PlayerAwards(player_id=pid)`, one call per player rather than per season since there's no league wide awards endpoint. Looped over the unique player ids with 100+ FGA in at least one season (pulled from `player_stats_basic_df`), not every player in the league.

`season_shots_df` / `close_game_shots_df`: `shotchartdetail.ShotChartDetail`, one call per player season, `context_measure_simple='FGA'`. The close game version adds `point_diff_nullable`, then gets filtered down further (period >= 4, minutes remaining < 5) to get the true clutch shot set.

`primary_defenders_df`: `leagueseasonmatchups.LeagueSeasonMatchups`, one call per player season, `off_player_id_nullable=player_id`, then takes the defender with the most `MATCHUP_TIME_SEC`.

`team_def_ratings`: `leaguedashteamstats.LeagueDashTeamStats`, one call per season, `measure_type_detailed_defense='Advanced'`.

`active_players_df`: `players.get_active_players()`, static local lookup, no network call, only used as a denominator for the volume filter.

Every loop that hits the API sleeps 0.6s between calls to stay under rate limits, and every fetch function checks for an existing parquet file first so nothing gets re pulled once it's cached.

## building `df`

Two merges. Advanced + basic season stats on `PLAYER_ID`/`SEASON`, inner join, since they come from separate endpoint calls and need stitching into one row per player season.

Then that result merges with the clutch stats, same keys, also inner. Both sides share column names (`TS_PCT`, `USG_PCT`, `GP`...) so the `_SEASON`/`_CLUTCH` suffixes are what make `TS_DELTA` possible later. `df` ends up being the table basically everything downstream references.

In [ ]:
adv_cols = ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'MIN',
            'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'EFG_PCT', 'TS_PCT',
            'USG_PCT', 'PIE', 'POSS', 'SEASON']
base_cols = ['PLAYER_ID', 'FGA', 'FG3A', 'FTA', 'PTS', 'PLUS_MINUS', 'REB', 'AST', 'SEASON']

# advanced + basic season stats, on player_id + season
season_df = player_stats_advanced_df[adv_cols].merge(
    player_stats_basic_df[base_cols], on=['PLAYER_ID', 'SEASON'], how='inner'
)

clutch_cols = ['PLAYER_ID', 'SEASON', 'GP', 'MIN', 'EFG_PCT', 'TS_PCT', 'USG_PCT',
               'NET_RATING', 'PIE', 'FGA']
clutch_slim = clutch_adv_df[clutch_cols]

# + clutch stats, this is the main df from here on
df = season_df.merge(clutch_slim, on=['PLAYER_ID', 'SEASON'], how='inner', suffixes=('_SEASON', '_CLUTCH'))

## `hook_context`, career average check

Merges on `PLAYER_ID` only, no season. `career_ts` is a full career average (groupby mean over all of `player_stats_advanced_df`). Point of this one is checking whether a player's clutch dip this season is normal for them or actually unusual, comparing to their own career TS%.

In [ ]:
hook_context = current_season.nsmallest(6, 'TS_DELTA')[
    ['PLAYER_ID', 'PLAYER_NAME', 'TS_PCT_SEASON', 'TS_PCT_CLUTCH', 'TS_DELTA', 'FGA_CLUTCH']
].sort_values('TS_DELTA').reset_index(drop=True)

career_ts = (player_stats_advanced_df[player_stats_advanced_df['PLAYER_ID'].isin(hook_context['PLAYER_ID'])]
             .groupby('PLAYER_ID')['TS_PCT'].mean().rename('CAREER_TS_PCT'))
# no SEASON key here on purpose, full career avg
hook_context = hook_context.merge(career_ts, on='PLAYER_ID')

## `matchup_analysis`, defender quality

First merge grabs the defender's own season `DEF_RATING`, reusing `player_stats_advanced_df` again but renamed so it joins on `DEF_PLAYER_ID` instead of `PLAYER_ID`. Left join since not every defender necessarily has a matching row.

Second merge puts the offensive player's `TS_DELTA` back next to their defender's rating, inner this time, keys are `PLAYER_ID`+`SEASON` again. For the "does a tougher primary defender explain the clutch drop" chart.

In [ ]:
defender_quality = player_stats_advanced_df[['PLAYER_ID', 'SEASON', 'DEF_RATING']].rename(
    columns={'PLAYER_ID': 'DEF_PLAYER_ID', 'DEF_RATING': 'DEFENDER_DEF_RATING'}
)

matchup_analysis = primary_defenders_df.merge(defender_quality, on=['DEF_PLAYER_ID', 'SEASON'], how='left')
matchup_analysis = matchup_analysis.merge(
    df[['PLAYER_ID', 'SEASON', 'TS_DELTA']], on=['PLAYER_ID', 'SEASON'], how='inner'
)

## `hr`, home vs road split

`home_clutch_df` and `road_clutch_df` have identical column names, so merging them with `_HOME`/`_ROAD` suffixes is what makes `TS_PCT_HOME` vs `TS_PCT_ROAD` something you can actually subtract.

Second merge just reattaches `PLAYER_NAME` and `group` from `df`, since the home/road tables don't carry those.

In [ ]:
hr_cols = ['PLAYER_ID', 'SEASON', 'GP', 'TS_PCT', 'NET_RATING', 'USG_PCT']

hr = (home_clutch_df[hr_cols]
      .merge(road_clutch_df[hr_cols], on=['PLAYER_ID', 'SEASON'], suffixes=('_HOME', '_ROAD')))
hr = hr[(hr['GP_HOME'] >= 8) & (hr['GP_ROAD'] >= 8)].copy()
# pull name + tier label back in, home/road tables don't have it
hr = hr.merge(df[['PLAYER_ID', 'SEASON', 'PLAYER_NAME', 'group']], on=['PLAYER_ID', 'SEASON'], how='inner')

## `shots_with_opponent`, case study section

`game_opponents` isn't a merge itself, it's built with a dict `.map()` (`team_lookup`) plus `np.where` to work out who the opponent was per game. The actual merges:

1. shots + `game_opponents` on `GAME_ID`+`PLAYER_NAME`, left join, attaches `opponent_abbr` to each shot.
2. that result + `team_def_ratings`, different column names on each side (`opponent_abbr` vs `TEAM_ABBREVIATION`), left join, attaches the opponent's season `DEF_RATING`.

End goal is checking whether shot making drops specifically against good defenses, at the individual shot level.

In [ ]:
shots_with_opponent = case_true_clutch_shots_v2.merge(
    game_opponents[['GAME_ID', 'PLAYER_NAME', 'opponent_abbr']],
    on=['GAME_ID', 'PLAYER_NAME'], how='left'
)

shots_with_opponent = shots_with_opponent.merge(
    team_def_ratings[['TEAM_ABBREVIATION', 'SEASON', 'DEF_RATING']],
    left_on=['opponent_abbr', 'SEASON'], right_on=['TEAM_ABBREVIATION', 'SEASON'], how='left'
)

## `wp1` / `wp1_matrix_df`, adding assist/rebound columns back in

`df` only kept shooting/usage columns from the advanced+clutch tables originally, so this goes back to `player_stats_advanced_df` and `clutch_adv_df` for `AST_PCT`/`REB_PCT` that got dropped earlier. Same two left merges done twice (once per chart, different base column subset). Checking whether a scoring decline in the clutch just shows up as more passing or rebounding instead.

In [ ]:
extra_season = player_stats_advanced_df[['PLAYER_ID', 'SEASON', 'AST_PCT', 'REB_PCT']]
extra_clutch = clutch_adv_df[['PLAYER_ID', 'SEASON', 'AST_PCT', 'REB_PCT']]

wp1 = df[['PLAYER_ID', 'SEASON', 'PLAYER_NAME', 'group', 'TS_DELTA']].merge(
    extra_season, on=['PLAYER_ID', 'SEASON'], how='left'
).merge(extra_clutch, on=['PLAYER_ID', 'SEASON'], how='left', suffixes=('_SEASON', '_CLUTCH'))

# wp1_matrix_df = same two merges again, just a different column subset of df going in

## `df_star`, pre fame vs post fame

`first_award_season` comes from grouping `all_awards_df` by `PERSON_ID` and taking the min season among the "big" award types. Merge is keyed on `PLAYER_ID` on the left against the index on the right (`right_index=True`, since `PERSON_ID` became the index after the groupby). Left join, only applied to the Star tier. Splits each star's seasons into pre/post their first major award.

In [ ]:
star_awards = all_awards_df[all_awards_df['DESCRIPTION'].isin(star_descriptions)]
first_award_season = star_awards.groupby('PERSON_ID')['SEASON'].min().rename('FIRST_AWARD_SEASON')

df_star = df[df['group'] == 'Star'].merge(first_award_season, left_on='PLAYER_ID', right_index=True, how='left')

## not merges, but related

`fetch_shots`, `fetch_primary_defenders`, and `fetch_team_def_ratings` all loop and `pd.concat` a bunch of same shaped frames together (one per API call) rather than joining on a key. Same with `matchup_sample`, which just stacks the top N players from each tier before the defender lookups above. Worth remembering these aren't joins if tracing where a column came from later.